# Análise macro: SELIC, IPCA e PIB

Um esboço de análise da política monetária brasileira com o `brazilfi`, cruzando três séries:

1. **SELIC meta** (Bacen, série SGS 432) — o juro básico definido pelo Copom.
2. **IPCA acumulado em 12 meses** (IBGE, agregado SIDRA 7060) — a inflação que o Copom persegue.
3. **PIB trimestral** (IBGE, agregado SIDRA 1620) — a atividade, medida pela variação em volume.

Com elas calculamos o **juro real ex-post** (SELIC descontada da inflação) e montamos um gráfico
que resume o ciclo de política monetária desde 2020.

**Requisitos:** `pip install "brazilfi[examples]"` (instala o matplotlib) e um Jupyter
(`pip install notebook`). Todas as fontes são públicas, sem credencial; cada célula de dados faz
uma chamada HTTP.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from brazilfi import Bacen, IBGE

bc = Bacen()
ibge = IBGE()

INICIO = "2020-01-01"  # o agregado 7060 do IPCA começa em jan/2020

## 1. SELIC meta (Bacen)

A série SGS 432 é a meta da SELIC definida pelo Copom. O Bacen a publica diariamente, mas o valor só
muda nas reuniões (a cada ~45 dias). Para cruzar com séries mensais, ficamos com o último valor de
cada mês. A API do SGS limita janelas de séries diárias a 10 anos, por isso passamos `start`.

In [ ]:
selic = (
    bc.selic(meta=True, start=INICIO)
    .to_dataframe()
    .resample("MS")
    .last()["value"]
    .rename("selic_meta")
)
selic.tail()

## 2. IPCA acumulado em 12 meses (IBGE)

O agregado 7060 do SIDRA traz o IPCA a partir de jan/2020. A variável 2265 é a variação acumulada em
12 meses, a medida que o Copom compara com a meta de inflação. Aqui usamos `ibge.agregado()`, o
acesso genérico a qualquer tabela do SIDRA.

In [ ]:
ipca_12m = (
    ibge.agregado(7060, 2265, last=120, name="IPCA acumulado 12 meses", unit="%")
    .to_dataframe()["value"]
    .rename("ipca_12m")
)
ipca_12m.tail()

## 3. PIB trimestral (IBGE)

O agregado 1620 é a série encadeada do índice de volume trimestral do PIB (média de 1995 = 100), sem
ajuste sazonal. A leitura usual é a variação contra o mesmo trimestre do ano anterior, que elimina a
sazonalidade: `pct_change(4)`.

> O SIDRA identifica trimestres como `AAAATT` (ex.: `202504` = 4º trimestre de 2025), e o `brazilfi`
> lê esse código como ano-mês. A ordem dos pontos está correta; só o rótulo é refeito abaixo.

In [ ]:
pib = ibge.pib(last=32).to_dataframe()["value"]
# 202504 → "2025Q4": reconstruímos o índice trimestral a partir de ano + "mês"
trimestres = pib.index.year.astype(str) + "Q" + pib.index.month.astype(str)
pib.index = pd.PeriodIndex(trimestres, freq="Q").to_timestamp()
pib_yoy = (pib.pct_change(4) * 100).dropna().rename("pib_yoy")
pib_yoy.tail()

## 4. Juro real ex-post

Juntamos SELIC e IPCA numa base mensal (o IPCA 12 meses fica em branco até nov/2020). O juro real é a SELIC descontada da inflação pela equação
de Fisher, `(1 + selic) / (1 + ipca) − 1`. Juro real alto e positivo indica política contracionista;
negativo, expansionista (como em 2020-21).

In [ ]:
df = pd.concat([selic, ipca_12m], axis=1)  # o IPCA 12 meses só existe a partir de dez/2020
df["juro_real"] = ((1 + df.selic_meta / 100) / (1 + df.ipca_12m / 100) - 1) * 100
df.tail(12).round(2)

## 5. Gráfico

Painel de cima: SELIC meta e IPCA 12 meses na mesma escala (% a.a.), com a meta de inflação fixada
pelo CMN e sua banda de tolerância de ±1,5 p.p. Painel de baixo: PIB, variação em volume contra o
mesmo trimestre do ano anterior.

In [ ]:
META_CMN = {2020: 4.0, 2021: 3.75, 2022: 3.5, 2023: 3.25, 2024: 3.0}  # 3% contínuo desde 2025
meta = pd.Series({pd.Timestamp(y, 1, 1): m for y, m in META_CMN.items()})
meta = meta.reindex(df.index, method="ffill")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, height_ratios=[3, 2])

ax1.fill_between(df.index, meta - 1.5, meta + 1.5, color="#c3c2b7", alpha=0.35,
                 label="banda de tolerância (±1,5 p.p.)")
ax1.step(df.index, meta, where="post", color="#52514e", lw=1, label="meta de inflação (CMN)")
ax1.plot(df.index, df.selic_meta, color="#2a78d6", lw=2, label="SELIC meta")
ax1.plot(df.index, df.ipca_12m, color="#eb6834", lw=2, label="IPCA 12 meses")
ax1.set_ylabel("% a.a.")
ax1.set_title("Política monetária: juros, inflação e atividade")
ax1.legend(loc="upper left", frameon=False)

cores = ["#2a78d6" if v >= 0 else "#e34948" for v in pib_yoy]
ax2.bar(pib_yoy.index, pib_yoy, width=80, align="edge", color=cores)
ax2.axhline(0, color="#52514e", lw=1)
ax2.set_ylabel("PIB, var. anual (%)")
ax2.set_xlim(pd.Timestamp(INICIO), df.index.max())

for ax in (ax1, ax2):
    ax.grid(axis="y", color="#e5e4df", lw=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()

## Leitura

- **2020-21:** SELIC em 2% com a inflação acelerando: juro real negativo, política expansionista
  para atravessar a pandemia. O PIB desaba e depois recupera.
- **2021-22:** o Copom leva a SELIC de 2% a 13,75% enquanto o IPCA 12 meses passa de 10%, bem
  acima do teto da banda. O juro real vira e fica fortemente positivo.
- **2023 em diante:** a inflação converge para a banda e começa o ciclo de cortes; o PIB cresce
  perto de 3% ao ano. Em 2024-25 a inflação volta a testar o teto, a SELIC sobe a 15% e os cortes só recomeçam em 2026.

Extensões naturais: trocar `ibge.agregado(7060, 2265)` por `bc.ipca(acum_12m=True)` (SGS 13522)
para estender a janela antes de 2020, ou incluir o CDI (`bc.cdi()`) e o câmbio (`bc.dolar()`).